# 03. Create Master Table & MLflow Logging
Merges structured and unstructured data into a single master Delta Table. Logs the dataset parameters to MLflow to satisfy the MLOps criteria.

In [ ]:
import mlflow

# --- Configuration ---
CATALOG = "main"
SCHEMA = "ayurgenix"
TABLE_CSV = f"{CATALOG}.{SCHEMA}.ayurgenix_structured"
TABLE_PDF = f"{CATALOG}.{SCHEMA}.ayurgenix_unstructured"
MASTER_TABLE = f"{CATALOG}.{SCHEMA}.ayurgenix_master"

In [ ]:
print("🔄 Merging datasets...")

csv_df = spark.table(TABLE_CSV)
pdf_df = spark.table(TABLE_PDF)

# Union the tables
master_df = csv_df.unionByName(pdf_df)
total_records = master_df.count()

print(f"✅ Total records combined: {total_records}")

In [ ]:
print("🧪 Logging to MLflow...")

# Start an MLflow run to show MLOps maturity
with mlflow.start_run(run_name="AyurGenix_Ingestion_Run"):
    mlflow.log_param("csv_record_count", csv_df.count())
    mlflow.log_param("pdf_chunk_count", pdf_df.count())
    mlflow.log_param("total_master_records", total_records)
    mlflow.log_param("target_catalog", CATALOG)
    mlflow.log_param("target_schema", SCHEMA)
    
    print("✅ Run logged to MLflow successfully.")

In [ ]:
print(f"💾 Saving final Master Delta Table: {MASTER_TABLE}...")

master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(MASTER_TABLE)

print("🏆 Ingestion Pipeline Complete!")
display(spark.table(MASTER_TABLE).limit(10))

**Next Step:** Proceed to Folder `02_aibi_dashboard` to visualize this master data.